In [14]:
print('trying out colab on vscode')
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

import sys

print("=" * 60)
print("🔍 SYSTEM CHECK")
print("=" * 60)

# Python version
print(f"Python Version: {sys.version.split()[0]}")

# PyTorch version
print(f"PyTorch Version: {torch.__version__}")

# CUDA (GPU support)
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Check GPU compute capability
    major, minor = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {major}.{minor}")

# Where are we running?
print(f"\nRunning on: Google Colab (remote)")

print("=" * 60)
print("✅ Setup verified - ready to code!")
print("=" * 60)

import time
device = torch.device('cuda')
print("🔥 GPU Performance Test")
print("=" * 50)

# Test matrix multiplication speed
sizes = [1000, 2000, 5000, 10000]

for size in sizes:
    # CPU test
    x_cpu = torch.randn(size, size)
    start = time.time()
    y_cpu = x_cpu @ x_cpu
    cpu_time = time.time() - start
    
    # GPU test
    x_gpu = torch.randn(size, size, device=device)
    torch.cuda.synchronize()  # Wait for GPU
    start = time.time()
    y_gpu = x_gpu @ x_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    
    speedup = cpu_time / gpu_time
    print(f"Matrix {size}×{size}:")
    print(f"  CPU: {cpu_time:.4f}s | GPU: {gpu_time:.4f}s | Speedup: {speedup:.1f}x")

print("=" * 50)
print("✅ GPU is fast and working perfectly!")

trying out colab on vscode
GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.83 GB
🔍 SYSTEM CHECK
Python Version: 3.12.12
PyTorch Version: 2.9.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Device: Tesla T4
GPU Memory: 15.83 GB
Compute Capability: 7.5

Running on: Google Colab (remote)
✅ Setup verified - ready to code!
🔥 GPU Performance Test
Matrix 1000×1000:
  CPU: 0.0140s | GPU: 0.0976s | Speedup: 0.1x
Matrix 2000×2000:
  CPU: 0.0968s | GPU: 0.0061s | Speedup: 15.8x
Matrix 5000×5000:
  CPU: 0.9570s | GPU: 0.0880s | Speedup: 10.9x
Matrix 10000×10000:
  CPU: 6.5126s | GPU: 0.5499s | Speedup: 11.8x
✅ GPU is fast and working perfectly!


Trying a simple example of a transformer model. 

In [8]:
import torch
import torch.nn as nn

In [9]:
class PatchEmbedding(nn.Module):
    """ 
    Convert image to patch embeddings.
    Example: 28 x 28 image with patch size of 7
    results in 16 patches of size 7 x 7, each flattened to a vector of size 49.

    """
    def __init__(self, img_size=28, patch_size=7, in_channels=1, embed_dim=64):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.projection = nn.Linear(patch_size * patch_size * in_channels, embed_dim)

    def forward(self, x):
        # x: (batch_size, channels, height, width) = [B, 1, 28, 28]

        batch__size = x.shape[0]

        # Unfold into patches: [B, 1, 28, 28] -> [B, 16, 49]
        patches = x.unfold(2, self.patch_size, self.patch_size) # unfold height
        print(patches.shape)
        patches = patches.unfold(3, self.patch_size, self.patch_size) # unfold width
        print(patches.shape)
        patches = patches.contiguous().view(batch__size, -1, self.patch_size * self.patch_size) # flatten patches
        print(patches.shape)

        # Project patches to embedding dimension: [B, 16, 49] -> [B, 16, 64]
        embeddings = self.projection(patches)
        print(embeddings.shape)
        return embeddings

In [10]:
patch_embed = PatchEmbedding(img_size=28, patch_size=7, in_channels=1, embed_dim=64)
x = torch.randn(8, 1, 28, 28)  # Example input: batch of 8 grayscale images of size 28x28
patches = patch_embed(x)
print(patches.shape)  # Expected output: [8, 16, 64]

print(patches)

torch.Size([8, 1, 4, 28, 7])
torch.Size([8, 1, 4, 4, 7, 7])
torch.Size([8, 16, 49])
torch.Size([8, 16, 64])
torch.Size([8, 16, 64])
tensor([[[ 1.8887e-01,  3.6866e-01, -5.2077e-01,  ...,  1.4782e-01,
          -6.9853e-01,  8.2930e-02],
         [-7.0993e-01,  4.8718e-01, -3.4821e-01,  ...,  3.7030e-01,
          -4.2592e-01,  2.6535e-01],
         [-4.5651e-01, -1.1512e+00, -4.5098e-02,  ...,  5.7436e-02,
          -8.5320e-01,  1.6332e-02],
         ...,
         [ 2.2487e-01,  1.1329e+00,  1.0413e+00,  ...,  9.7145e-01,
           8.1362e-01, -2.1531e-02],
         [ 4.0684e-01,  7.5340e-01,  2.1180e-01,  ..., -2.2983e-01,
           6.1863e-01, -5.1557e-02],
         [ 3.4136e-01, -8.0423e-02, -6.0903e-01,  ...,  8.1435e-01,
           2.2467e-01, -2.9429e-01]],

        [[ 2.8544e-01, -8.4561e-01, -6.4769e-01,  ...,  1.0582e+00,
           1.0111e-01, -3.7388e-01],
         [ 1.2679e-01, -3.8709e-01, -5.2833e-01,  ..., -1.0743e+00,
          -3.7557e-01, -5.8887e-01],
         [ 2

In [11]:
# positional embedding

class PositionalEmbedding(nn.Module):
    '''
    Add leardned positional embeddings to patches. 
    Each of the 16 patches gets a unique position vector.
    '''
    def __init__(self, n_patches = 16, embedding_dim=64):
        super().__init__()
        #Learnable position embeddings
        self.position_embeddings = nn.Parameter(torch.randn(1, n_patches, embedding_dim))

    def forward(self, x):
        # x: [batch, n_patches, embedding_dim]
        # add position embeddings (broadcast across batch)
        return x + self.position_embeddings 